# A10：Memory Architecture — 四種記憶類型完整實作

> **對應 Blog：** FDE 面試準備指南（十四）AI Agent Memory 架構設計  
> **核心目標：** 實作四種記憶類型，建立完整的 Memory Retrieval + Update 流程，理解每種記憶解決的具體問題。

## 面試情境

> 「你的 Agent 需要記住用戶的偏好、過去幾個月的對話，以及如何執行特定任務。你怎麼設計記憶系統？為什麼需要不同類型的記憶？」

## 學習路徑

| Part | 記憶類型 | 解決什麼問題 | 儲存方式 |
|------|----------|-------------|----------|
| 1 | Working Memory | 當前對話臨時狀態 | Context Window |
| 2 | Episodic Memory | 記住過去發生什麼 | 向量 DB |
| 3 | Semantic Memory | 記住這個人是誰 | 結構化 DB |
| 4 | Procedural Memory | 知道怎麼做某件事 | Few-shot Prompt |
| 5 | 完整 Memory Pipeline | 組合使用 | |
| 6 | 四大工程挑戰 | 衝突/過時/隱私/精確性 | |
| 7 | 面試白板語言 | | |

In [ ]:
import time
import json
import uuid
import asyncio
import math
from typing import Any, Optional
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from collections import defaultdict

try:
    from dotenv import load_dotenv
    load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    LLM_AVAILABLE = True
    print("✅ LLM 可用")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，使用模擬輸出")

print("環境就緒")

## 問題背景：為什麼需要四種記憶？

**沒有記憶的 Agent：**

In [ ]:
print("""
沒有記憶的 Agent 的問題：

Session 1（星期一）：
  User: "我主要用 Python，偏好簡短的回答"
  Agent: "好的！" （記不住）

Session 2（星期四）：
  User: "幫我寫一個排序函數"
  Agent: "您好！請問您用哪種程式語言？"
          ↑ 明明說過了，還在問

Session 3（三週後）：
  User: "上次那個 dbt 問題解決了嗎？"
  Agent: "抱歉，我不記得我們之前的對話。"
          ↑ 用戶很沮喪

為什麼「把所有對話都記住」也不行？
  10K 用戶 × 每天 10 輪 × 365 天 = 3,650 萬條對話
  全部塞進 context → 超過 context window
  全部召回 → 大量不相關資訊污染 context

解法：按用途設計四種記憶類型，各司其職
""")

## Part 1：Working Memory — 當前對話的臨時狀態

In [ ]:
class WorkingMemory:
    """
    Working Memory：LLM Context Window 的管理
    生命週期：當次對話（Session 結束即清空）
    解決問題：當前對話的臨時狀態和推理上下文
    """
    
    # Context 各區塊的 Token 預算
    BUDGET = {
        "system_prompt": 500,
        "semantic_memory": 300,    # User Profile
        "procedural_examples": 1000,
        "episodic_recall": 2000,   # 召回的歷史記憶
        "conversation": 4000,       # 當前對話歷史（動態）
        "output_reserve": 2000,
    }
    
    def __init__(self, max_tokens: int = 128_000):
        self.max_tokens = max_tokens
        self.conversation_history: list[dict] = []
        self.session_id = str(uuid.uuid4())[:8]
    
    def add_turn(self, role: str, content: str):
        self.conversation_history.append({"role": role, "content": content})
    
    def assemble_context(
        self,
        system_prompt: str,
        semantic_memory_str: str,
        episodic_recall: list[str],
        procedural_examples: str,
        current_query: str
    ) -> str:
        """
        組裝完整 Context（注意排列順序！）
        Lost-in-the-Middle 對策：關鍵資訊放靠近結尾
        """
        
        # 順序設計：高注意力（開頭/結尾）放最重要的資訊
        parts = [
            f"[System]\n{system_prompt}",
            f"[User Profile - 每次載入]\n{semantic_memory_str}",
            f"[How-to Examples - 程序記憶]\n{procedural_examples}",
        ]
        
        if episodic_recall:
            parts.append("[相關歷史記憶 - 按相似度召回]")
            for i, mem in enumerate(episodic_recall, 1):
                parts.append(f"  記憶 {i}: {mem}")
        
        # 最近對話歷史（靠近結尾，注意力高）
        if self.conversation_history:
            parts.append("[最近對話]")
            for msg in self.conversation_history[-6:]:  # 最近 3 輪
                parts.append(f"  {msg['role']}: {msg['content'][:100]}")
        
        # 當前 Query 在最後（最高注意力）
        parts.append(f"\n記住以上所有資訊，請回答：\n{current_query}")
        
        return "\n\n".join(parts)
    
    def token_estimate(self, text: str) -> int:
        return len(text.split()) * 4 // 3


wm = WorkingMemory()
wm.add_turn("user", "我想查 Q4 銷售數據")
wm.add_turn("assistant", "好的，讓我查詢一下...")

test_context = wm.assemble_context(
    system_prompt="你是一個數據分析助理。",
    semantic_memory_str="用戶：Alice｜職稱：Data Engineer｜偏好：Python 和 BigQuery",
    episodic_recall=["2026-05: 討論了 Q3 銷售下降的原因", "2026-04: 建立了 dbt 報表模型"],
    procedural_examples="如何查詢 BigQuery：...\n如何解讀銷售指標：...",
    current_query="Q4 的銷售總額是多少？"
)

print("Working Memory Context 組裝：")
print("=" * 55)
print(test_context)
print(f"\n總長度：{wm.token_estimate(test_context)} tokens（估算）")
print("\n⭐ 注意排列順序：System → Profile → Examples → History → Query")
print("   關鍵資訊（Profile, 最近對話）靠近開頭和結尾（高注意力區）")

## Part 2：Episodic Memory — 記住過去發生什麼

In [ ]:
@dataclass
class EpisodicMemoryEntry:
    """一條情節記憶：記錄一次對話中發生的事"""
    memory_id: str
    user_id: str
    content: str           # 摘要後的記憶內容
    importance: float      # 0.0 ~ 1.0，越高越重要
    timestamp: float
    session_id: str
    # 生產中這裡存 embedding vector
    _keywords: list[str] = field(default_factory=list)  # 用於模擬相似度搜尋


class EpisodicMemoryStore:
    """
    Episodic Memory：向量化的對話歷史
    生命週期：跨 Session，可衰減
    解決問題：「幾個月前說過的話」
    生產實作：Vertex AI Vector Search / pgvector
    """
    
    def __init__(self):
        self.memories: list[EpisodicMemoryEntry] = []
    
    def store(self, user_id: str, content: str, importance: float,
              session_id: str, keywords: list[str] = None) -> EpisodicMemoryEntry:
        """非同步儲存記憶（不在請求路徑上）"""
        entry = EpisodicMemoryEntry(
            memory_id=str(uuid.uuid4())[:8],
            user_id=user_id,
            content=content,
            importance=importance,
            timestamp=time.time(),
            session_id=session_id,
            _keywords=keywords or content.lower().split()
        )
        self.memories.append(entry)
        return entry
    
    def recall(self, user_id: str, query: str, top_k: int = 3,
               importance_threshold: float = 0.3) -> list[EpisodicMemoryEntry]:
        """
        按語意相似度召回相關記憶
        生產中：Embedding + cosine similarity
        這裡：詞集合 Jaccard 相似度模擬
        """
        query_words = set(query.lower().split())
        user_memories = [m for m in self.memories 
                        if m.user_id == user_id and m.importance >= importance_threshold]
        
        def score(mem: EpisodicMemoryEntry) -> float:
            mem_words = set(mem._keywords)
            if not (query_words | mem_words):
                return 0
            sim = len(query_words & mem_words) / len(query_words | mem_words)
            # 重要性加權 + 時間衰減（舊記憶稍微降權）
            age_days = (time.time() - mem.timestamp) / 86400
            recency = math.exp(-age_days / 90)  # 90 天半衰期
            return sim * 0.6 + mem.importance * 0.3 + recency * 0.1
        
        scored = sorted(user_memories, key=score, reverse=True)
        return scored[:top_k]
    
    def assess_importance(self, content: str) -> float:
        """
        判斷記憶重要性（生產中用 LLM 判斷）
        高重要性：偏好聲明、未解決問題、重要決策
        低重要性：閒聊、例行查詢
        """
        high_signals = ["偏好", "不喜歡", "問題", "錯誤", "決定", "選擇", "需要", "重要"]
        low_signals = ["謝謝", "好的", "了解", "沒關係"]
        
        content_lower = content.lower()
        if any(s in content_lower for s in low_signals):
            return 0.2
        if any(s in content_lower for s in high_signals):
            return 0.8
        return 0.5


# 測試
episodic = EpisodicMemoryStore()

memories_to_store = [
    "用戶報告 dbt 模型跑太慢，原因是缺少 partition，已建議加 DATE 分區",
    "用戶偏好 Python，不喜歡用 Java，偏好簡短清晰的回答",
    "Q3 銷售下降原因分析：主要是台中區域客戶流失，建議加強售後服務",
    "用戶說好的，謝謝",  # 低重要性
    "用戶確認採用 BigQuery 方案，而非 Redshift",
]

print("=" * 55)
print("Episodic Memory 儲存 + 召回測試")
print("=" * 55)

for content in memories_to_store:
    importance = episodic.assess_importance(content)
    entry = episodic.store(
        user_id="alice",
        content=content,
        importance=importance,
        session_id=f"session_{len(episodic.memories)}"
    )
    icon = "⭐" if importance >= 0.7 else ("📝" if importance >= 0.4 else "💬")
    print(f"  {icon} importance={importance:.1f} | {content[:60]}...")

# 召回測試
queries = [
    "dbt 的問題解決了嗎？",
    "我的程式語言偏好是什麼？",
    "BigQuery 的決策",
]

print("\n召回測試：")
for query in queries:
    recalled = episodic.recall("alice", query, top_k=2)
    print(f"\n  Query: '{query}'")
    for mem in recalled:
        print(f"    ↳ [importance={mem.importance:.1f}] {mem.content[:70]}")

## Part 3：Semantic Memory — 記住這個人是誰

In [ ]:
@dataclass
class UserProfile:
    """
    Semantic Memory：結構化的 User Profile
    生命週期：持久化，主動更新
    解決問題：記住「這個人是什麼樣的人、偏好什麼」
    生產實作：PostgreSQL / Firestore
    """
    user_id: str
    
    # 身份（相對穩定）
    name: str = ""
    role: str = ""             # Data Engineer
    department: str = ""
    
    # 偏好（從對話中學習，有半衰期）
    preferred_language: str = "Python"
    response_style: str = "簡潔"  # 簡潔 | 詳細
    technical_level: str = "intermediate"  # beginner | intermediate | advanced
    
    # 工作背景（頻繁更新）
    tools: list[str] = field(default_factory=list)
    active_projects: list[str] = field(default_factory=list)
    open_issues: list[dict] = field(default_factory=list)
    
    # Meta
    last_updated: float = field(default_factory=time.time)
    confidence_scores: dict = field(default_factory=dict)  # 欄位 → 信心分數
    
    def to_prompt_str(self) -> str:
        """轉換成注入 Context 的字串"""
        issues_str = "; ".join(
            f"{i['issue']}（{i.get('since', '?')}起）"
            for i in self.open_issues
        ) or "無"
        return f"""用戶：{self.name or '未知'} | 職稱：{self.role or '未知'}
技術偏好：{self.preferred_language} | 回答風格：{self.response_style} | 程度：{self.technical_level}
工具：{', '.join(self.tools) or '未知'}
進行中專案：{', '.join(self.active_projects) or '無'}
未解決問題：{issues_str}"""
    
    def update_from_conversation(self, key: str, value: Any, confidence: float = 0.7):
        """從對話中更新 Profile（帶 confidence score）"""
        current_confidence = self.confidence_scores.get(key, 0)
        
        # 只在新資訊信心分數 > 現有信心時更新
        if confidence > current_confidence or current_confidence < 0.3:
            if hasattr(self, key):
                setattr(self, key, value)
                self.confidence_scores[key] = confidence
                self.last_updated = time.time()
                print(f"  📝 Profile 更新：{key} = {value}（信心={confidence:.1f}）")
            else:
                print(f"  ⚠️  未知欄位：{key}")
        else:
            print(f"  ⏭️  跳過更新 {key}（現有信心={current_confidence:.1f} >= 新信心={confidence:.1f}）")


class SemanticMemoryStore:
    def __init__(self):
        self._profiles: dict[str, UserProfile] = {}
    
    def get_or_create(self, user_id: str) -> UserProfile:
        if user_id not in self._profiles:
            self._profiles[user_id] = UserProfile(user_id=user_id)
        return self._profiles[user_id]
    
    def get_stale_fields(self, profile: UserProfile, 
                          decay_days: dict = None) -> list[str]:
        """找出過期需要重確認的欄位"""
        if decay_days is None:
            decay_days = {
                "technical_level": 90,   # 90 天重確認
                "active_projects": 30,   # 30 天重確認
                "open_issues": 14,       # 14 天重確認
            }
        stale = []
        age_days = (time.time() - profile.last_updated) / 86400
        for field_name, max_days in decay_days.items():
            if age_days > max_days:
                stale.append(field_name)
        return stale


# 測試
semantic = SemanticMemoryStore()
alice = semantic.get_or_create("alice")

print("=" * 55)
print("Semantic Memory 測試")
print("=" * 55)

# 初始化 Profile（可能從 onboarding 表單取得）
alice.name = "Alice"
alice.role = "Data Engineer"
alice.tools = ["Python", "dbt", "BigQuery"]
alice.active_projects = ["data-pipeline-v2"]
alice.open_issues = [{"issue": "dbt 模型跑太慢", "since": "2026-05"}]

print("\n初始 Profile：")
print(alice.to_prompt_str())

# 從對話中學習並更新
print("\n從對話中更新 Profile：")
alice.update_from_conversation("technical_level", "advanced", confidence=0.85)
alice.update_from_conversation("technical_level", "beginner", confidence=0.4)  # 信心低，不更新
alice.update_from_conversation("response_style", "詳細", confidence=0.7)

print("\n更新後 Profile：")
print(alice.to_prompt_str())

## Part 4：Procedural Memory — 知道怎麼做某件事

In [ ]:
class ProceduralMemory:
    """
    Procedural Memory：Few-shot examples（示範如何做某件事）
    生命週期：最持久，在模型層（Few-shot in Prompt 或 Fine-tuning）
    解決問題：「知道怎麼做某類任務」
    
    兩種實作：
    1. Few-shot in Prompt（靈活，但佔 token）
    2. Fine-tuning（不佔 token，但更新成本高）
    """
    
    def __init__(self):
        # 按任務類型儲存 Few-shot 範例
        self._examples: dict[str, list[dict]] = defaultdict(list)
    
    def add_example(self, task_type: str, input_example: str,
                     output_example: str, note: str = ""):
        self._examples[task_type].append({
            "input": input_example,
            "output": output_example,
            "note": note
        })
    
    def get_examples_for_prompt(self, task_type: str, max_examples: int = 2) -> str:
        """組裝成 Few-shot Prompt 的格式"""
        examples = self._examples.get(task_type, [])[:max_examples]
        if not examples:
            return ""
        lines = [f"以下是 {task_type} 的示範："]
        for i, ex in enumerate(examples, 1):
            lines.append(f"\n範例 {i}：")
            lines.append(f"  輸入：{ex['input']}")
            lines.append(f"  輸出：{ex['output']}")
            if ex["note"]:
                lines.append(f"  說明：{ex['note']}")
        return "\n".join(lines)


proc = ProceduralMemory()

# 儲存「如何回答資料查詢問題」的示範
proc.add_example(
    task_type="data_query",
    input_example="Q4 2025 的銷售總額是多少？",
    output_example="""根據 BigQuery 查詢結果，Q4 2025 銷售總額為 $4,200,000 TWD。
較 Q3 成長 12%。主要成長來自北部區域（+18%）。""",
    note="先給數字，再給比較，最後給洞察"
)

proc.add_example(
    task_type="data_query",
    input_example="哪個產品線銷售最好？",
    output_example="""Top 3 產品線（Q4 2025）：
1. 企業軟體授權：$2.1M（50%）
2. 訂閱服務：$1.3M（31%）
3. 技術支援：$0.8M（19%）""",
    note="用排名格式，附比例"
)

proc.add_example(
    task_type="code_review",
    input_example="請幫我 review 這個 SQL 查詢",
    output_example="""查看了你的 SQL，發現三個可以改善的地方：
1. [效能] WHERE 子句沒有使用索引欄位
2. [可讀性] 子查詢可以改用 CTE 更清晰
3. [正確性] JOIN 條件可能產生重複列""",
    note="先講重要性，再給具體建議"
)

print("=" * 55)
print("Procedural Memory — Few-shot Examples")
print("=" * 55)

print("\n資料查詢的 Few-shot Prompt：")
print(proc.get_examples_for_prompt("data_query"))

print("\n\n💡 Procedural Memory 的兩種選項：")
print("""
Few-shot in Prompt（目前的方案）：
  優點：靈活，可以快速更新範例
  缺點：佔用 token（每個範例 ~100-300 tokens）
  適合：任務樣式還在演進的階段

Fine-tuning（進階方案）：
  優點：不佔 token，回應更一致
  缺點：更新成本高（需要重新 fine-tune）
  適合：任務樣式穩定後的生產優化
""")

## Part 5：完整 Memory Pipeline — 組合四種記憶

In [ ]:
class MemoryPipeline:
    """
    整合四種記憶的完整 Pipeline
    
    請求路徑（同步）：
      1. 載入 Semantic Memory（User Profile）
      2. 召回 Episodic Memory（相關歷史）
      3. 組裝 Working Memory（Context）
      4. 呼叫 LLM（帶 Procedural Memory 的 Few-shot）
    
    非同步路徑（對話結束後）：
      5. 更新 Semantic Memory（Profile 學習）
      6. 儲存 Episodic Memory（本次對話摘要）
    """
    
    def __init__(self):
        self.semantic = SemanticMemoryStore()
        self.episodic = EpisodicMemoryStore()
        self.procedural = proc  # 重用上面建立的
        self.working = WorkingMemory()
    
    async def process_request(self, user_id: str, query: str, task_type: str = "general") -> dict:
        """同步路徑：處理用戶請求"""
        
        print(f"\n[Memory Pipeline] 處理請求：'{query}'")
        
        # Step 1: 載入 Semantic Memory
        profile = self.semantic.get_or_create(user_id)
        print(f"  ① Semantic Memory 載入：{profile.name or user_id}")
        
        # Step 2: 召回 Episodic Memory
        recalled_memories = self.episodic.recall(user_id, query, top_k=3)
        print(f"  ② Episodic Memory 召回：{len(recalled_memories)} 條相關記憶")
        episodic_str = [m.content for m in recalled_memories]
        
        # Step 3: 組裝 Working Memory（Context）
        procedural_str = self.procedural.get_examples_for_prompt(task_type, max_examples=1)
        
        context = self.working.assemble_context(
            system_prompt="你是一個專業的數據分析助理，協助 Data Engineer 解決問題。",
            semantic_memory_str=profile.to_prompt_str(),
            episodic_recall=episodic_str,
            procedural_examples=procedural_str,
            current_query=query
        )
        print(f"  ③ Working Memory 組裝：{len(context.split())*4//3} tokens（估算）")
        
        # Step 4: 呼叫 LLM（模擬）
        if LLM_AVAILABLE:
            response = await llm.ainvoke([HumanMessage(content=context)])
            answer = response.content
        else:
            answer = f"[模擬回答] 根據您的偏好（{profile.preferred_language}）和記憶，{query[:20]}..."
        
        print(f"  ④ LLM 回答完成")
        
        # 更新 Working Memory 歷史
        self.working.add_turn("user", query)
        self.working.add_turn("assistant", answer)
        
        return {"answer": answer, "context_used": {
            "semantic": bool(profile.name),
            "episodic_count": len(recalled_memories),
            "procedural": bool(procedural_str)
        }}
    
    async def post_session_update(self, user_id: str, session_id: str,
                                    conversation: list[dict]):
        """非同步路徑：對話結束後更新記憶（不在請求路徑上）"""
        print(f"\n[非同步更新] 對話結束，更新記憶...")
        
        profile = self.semantic.get_or_create(user_id)
        
        # 提取重要資訊更新 Semantic Memory
        # 生產中：呼叫 LLM 提取結構化資訊
        combined_text = " ".join(m["content"] for m in conversation)
        
        # 簡單規則（生產中用 LLM）
        if "Python" in combined_text:
            profile.update_from_conversation("preferred_language", "Python", 0.8)
        
        # 儲存對話摘要到 Episodic Memory
        summary = combined_text[:200] + "..."  # 生產中用 LLM 摘要
        importance = self.episodic.assess_importance(summary)
        entry = self.episodic.store(user_id, summary, importance, session_id)
        print(f"  ✅ Episodic 儲存：importance={importance:.1f}")
        print(f"  ✅ 非同步更新完成（不影響請求路徑延遲）")


# 完整流程測試
pipeline = MemoryPipeline()

# 設定 Alice 的 Profile
alice_profile = pipeline.semantic.get_or_create("alice")
alice_profile.name = "Alice"
alice_profile.role = "Data Engineer"
alice_profile.tools = ["Python", "dbt", "BigQuery"]

# 先存一些歷史記憶
pipeline.episodic.store("alice", "dbt 模型優化，加了 DATE 分區後效能提升 80%",
                         0.8, "session_old", ["dbt", "分區", "效能"])
pipeline.episodic.store("alice", "Q3 銷售分析完成，發現台中區域流失最嚴重",
                         0.7, "session_old2", ["Q3", "銷售", "分析"])

# 處理新請求
result = await pipeline.process_request(
    user_id="alice",
    query="Q4 的銷售數據在哪裡查？有沒有比 Q3 好？",
    task_type="data_query"
)

print(f"\n回答: {result['answer'][:200]}")
print(f"記憶使用：{result['context_used']}")

# 非同步更新
await pipeline.post_session_update(
    user_id="alice",
    session_id="session_new",
    conversation=pipeline.working.conversation_history
)

## Part 6：四大工程挑戰

In [ ]:
print("""
四大工程挑戰：
═════════════════════════════════════════════════════

挑戰 1：記憶衝突（Conflict）
  問題：Profile 記錄 2026-01「技術等級：初學者」
        用戶現在已經是中級工程師了
  解法：
    - 新記憶優先（衝突時以最新記錄為準）
    - Confidence score（多次確認才更新 profile）
    - 主動重確認（6 個月未更新的欄位，主動詢問）

挑戰 2：記憶過時（Staleness）
  問題：不同欄位的「半衰期」不同
  Decay rules：
    技術偏好    → 90 天：對話中有新工具提及時重確認
    活躍專案    → 30 天：專案名稱長時間不出現時主動關閉
    未解決問題  → 14 天：問題被標記解決後自動關閉
    聯絡資訊    → 365 天：用戶主動更新

挑戰 3：隱私權（Privacy）
  GDPR 合規設計：
    GET  /memory/{user_id}          → 讓用戶看到 Agent 記了什麼
    DELETE /memory/{user_id}        → 全部刪除（right to erasure）
    DELETE /memory/{user_id}/episodes → 只刪對話歷史
    PUT /memory/{user_id}/profile   → 用戶直接修改自己的 profile

挑戰 4：召回精確性 vs 完整性
  Precision vs Recall 的 trade-off：
    threshold = 0.95（高精確）: 相關，但可能漏掉有用記憶
    threshold = 0.80（高召回）: 完整，但可能帶入不相關記憶
  建議：
    重要記憶（importance > 0.8）→ threshold 0.80（寧可多召回）
    一般記憶（importance < 0.5）→ threshold 0.92（只召回高相關）
═════════════════════════════════════════════════════
""")

## Part 7：面試白板語言 + 選型框架

In [ ]:
print("""
面試答題框架：
─────────────────────────────────────────────────────
記憶系統設計的核心問題是：
讓無狀態的 LLM 表現得像是「記得你」——
同時控制成本（不把所有歷史塞進 context）。

四種記憶各解決一個具體問題：

Working Memory（Context Window）：
  解決：當前對話的臨時推理
  設計：精心設計 context 排列順序（Lost-in-the-Middle）
  關鍵：System Prompt 和 Current Query 放開頭/結尾

Episodic Memory（Vector DB）：
  解決：「幾個月前說過的話」
  設計：按語意相似度召回，帶 importance score 和時間衰減
  關鍵：Memory Update 是非同步的（不在請求路徑上）

Semantic Memory（Structured DB）：
  解決：記住這個人的偏好、背景
  設計：User Profile + confidence score + decay rules
  關鍵：每次對話都載入，但不主動問用戶（從對話中學習）

Procedural Memory（Few-shot/Fine-tuning）：
  解決：「知道怎麼做某類任務」
  設計：先用 Few-shot，任務穩定後再考慮 Fine-tuning
  關鍵：放在 System Prompt 的 Few-shot 區，不佔主要 context

實作順序建議：
  Working → Semantic → Episodic → Procedural
  從最小可行的開始，不要一開始就建所有四種
─────────────────────────────────────────────────────
""")

print("選型決策框架：")
print(f"{'需求':<35} {'推薦記憶類型':<25} 儲存")
print("-" * 75)
decisions = [
    ("只需要當次對話的狀態",             "Working Memory only",      "Context Window"),
    ("記住用戶是誰、偏好什麼",            "+ Semantic Memory",         "PostgreSQL/Firestore"),
    ("記住幾個月前說過什麼",              "+ Episodic Memory",         "Vector DB"),
    ("跨用戶共享的最佳實踐",              "+ Procedural Memory",       "Few-shot in Prompt"),
    ("需要高度一致的任務執行方式",        "Procedural → Fine-tuning",  "Model 層"),
]
for need, memory_type, storage in decisions:
    print(f"  {need:<35} {memory_type:<25} {storage}")